In [1]:
import re
import os 
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from pyhive import presto
from datetime import date
from collections import Counter

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'

conn=presto.connect('presto-gateway.processing.data.production.internal',presto_port,username)

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


pd.set_option('display.max_columns', None)

## Overall

In [2]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        -- event_type,
        -- click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1
        )
    ORDER BY 1,2,3

"""

overall_sports=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62376/2332381144.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_sports=pd.read_sql(overall_query,conn)


In [10]:
overall_sports

,test_group,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,49649,146295,100582,68.75,85.58,83,108,255,159,10,0.59,10.66,16.39,3.16,860,15601,23981,4628
1,test,49282,148749,102576,68.96,85.71,66,108,245,157,10,0.63,10.45,16.32,3.20,941,15538,24271,4761


In [11]:
overall_sports.to_clipboard(index=False)

In [5]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        event_type,
        click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3

"""

overall_sportsl1=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62376/3688866530.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_sportsl1=pd.read_sql(overall_query,conn)


In [12]:
overall_sportsl1

,test_group,event_type,click_type,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,49649,146295,100582,68.75,85.58,83,108,255,159,10,0.59,10.66,16.39,3.16,860,15601,23981,4628
1,test,viewed,close_click,36440,104935,72217,68.82,85.41,67,109,247,158,10,0.67,10.66,16.16,3.25,698,11190,16957,3415
2,test,viewed,play_click,2178,4092,2498,61.05,81.84,39,97,203,129,10,0.90,13.42,20.33,3.84,37,549,832,157
3,test,viewed,None,1396,2740,1784,65.11,86.75,41,108,183,141,9,0.84,8.94,21.13,3.43,23,245,579,94
4,test,None,None,15462,37054,26100,70.44,86.90,75,106,248,159,10,0.51,9.61,16.01,2.98,188,3561,5931,1104


In [13]:
overall_sportsl1.to_clipboard(index=False)

In [64]:
overall_sports.to_csv('overall_sports.csv',index=False)

## Pre-post

In [7]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
    and event_props_campaign_id = '1725971334'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        -- event_type,
        -- click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1
        )
    ORDER BY 1,2,3

"""

overall_sportspre=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62376/3545597203.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_sportspre=pd.read_sql(overall_query,conn)


In [14]:
overall_sportspre

,test_group,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,48765,154753,90750,58.64,74.62,67,117,259,150,11,0.66,18.21,15.61,6.51,1026,28176,24158,10076
1,test,48702,156637,92985,59.36,75.22,74,117,259,150,12,0.70,17.73,15.47,6.34,1096,27769,24234,9935


In [15]:
overall_sportspre.to_clipboard(index=False)

## COBRA TTC 

In [26]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        -- event_type,
        -- click_type,
        cobra_ttc_bucket,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,

            CASE 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec <= 15 THEN 'a. 0-15 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 16 AND 30 THEN 'b. 16-30 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 31 AND 60 THEN 'c. 31-60 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 61 AND 180 THEN 'd. 61-180 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec > 180 THEN 'e. 180+ sec' 
            WHEN modified_order_status != 'COBRA' THEN 'f. NO COBRA'
            ELSE 'Check' END cobra_ttc_bucket,
            
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)

           and click_type = 'play_click'
           
        GROUP BY 1,2
        )
    ORDER BY 1,2

"""

overall_sports_ttc=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62376/2534316204.py:193: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_sports_ttc=pd.read_sql(overall_query,conn)


In [27]:
overall_sports_ttc

,test_group,cobra_ttc_bucket,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,test,a. 0-15 sec,46,52,0,0.00,0.00,NaN,11.0,NaN,11,NaN,0.00,100.0,0.00,0.00,0,52,0,0
1,test,b. 16-30 sec,56,58,0,0.00,0.00,NaN,22.0,NaN,22,NaN,0.00,100.0,0.00,0.00,0,58,0,0
2,test,c. 31-60 sec,61,69,0,0.00,0.00,NaN,47.0,NaN,47,NaN,0.00,100.0,0.00,0.00,0,69,0,0
3,test,d. 61-180 sec,175,258,0,0.00,0.00,NaN,112.0,NaN,112,NaN,0.00,100.0,0.00,0.00,0,258,0,0
4,test,e. 180+ sec,95,112,0,0.00,0.00,NaN,225.0,NaN,225,NaN,0.00,100.0,0.00,0.00,0,112,0,0
5,test,f. NO COBRA,2080,3543,2498,70.51,94.52,39.0,NaN,203.0,191,10.0,1.04,0.0,23.48,4.43,37,0,832,157


In [28]:
overall_sports_ttc.to_clipboard(index=False)

## other

In [82]:
overall_query_daywise=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
                from 
                    clevertap.clevertap_customer_order_activity
                where 
                    yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
                    AND order_activity_source = 'appOpen'
                    and current_city = 'Hyderabad'
                    and platform='Android'
                    and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        yyyymmdd,
        test_group,
        event_type,
        click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            yyyymmdd,
            test_group,
            event_type,
            click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

overall_sports_daywise=pd.read_sql(overall_query_daywise,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/2737540206.py:181: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_sports_daywise=pd.read_sql(overall_query_daywise,conn)


In [83]:
overall_sports_daywise

,yyyymmdd,test_group,event_type,click_type,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,20240911,control,None,None,14510,24860,17050,68.58,84.72,95.0,106,250,153,10,0.54,11.27,15.70,3.46,135,2802,3904,861
1,20240911,test,viewed,close_click,7130,13061,8826,67.58,84.83,90.0,112,222,152,10,0.67,10.99,16.79,3.51,88,1436,2193,458
2,20240911,test,viewed,play_click,306,637,362,56.83,78.34,46.0,99,152,119,11,1.41,15.38,21.35,4.87,9,98,136,31
3,20240911,test,viewed,None,216,391,257,65.73,85.93,NaN,91,271,169,10,0.00,9.97,19.95,4.09,0,39,78,16
4,20240911,test,None,None,7048,11347,8045,70.90,86.92,84.0,104,249,158,10,0.43,9.61,15.59,3.03,49,1091,1769,344
5,20240912,control,None,None,14706,24791,17203,69.39,85.73,80.0,111,254,161,10,0.61,10.30,15.95,3.36,150,2554,3955,832
6,20240912,test,viewed,close_click,10648,18566,12831,69.11,84.97,72.0,107,250,151,10,0.81,10.75,15.50,3.46,151,1996,2878,643
7,20240912,test,viewed,play_click,405,679,430,63.33,81.44,22.0,100,247,110,10,0.59,13.99,17.97,3.98,4,95,122,27
8,20240912,test,viewed,None,289,508,363,71.46,90.55,35.0,127,248,159,9,0.59,6.89,18.31,1.97,3,35,93,10
9,20240912,test,None,None,3428,5547,3977,71.70,87.29,55.0,106,229,153,10,0.50,9.00,15.05,3.21,28,499,835,178


In [85]:
overall_sports_daywise.to_clipboard(index=False)

In [67]:
overall_sports_daywise.to_csv('overall_sports_daywise.csv',index=False)

In [87]:
session_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android' 
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971334'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        event_type,
        click_type,
        time_period,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

session_sports=pd.read_sql(session_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/2198619014.py:187: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  session_sports=pd.read_sql(session_query,conn)


In [88]:
session_sports

,test_group,event_type,click_type,time_period,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,afternoon,25543,48700,36083,74.09,90.21,74.0,104,251,166,9,0.37,7.34,15.71,2.09,178,3576,7653,1016
1,control,None,None,evening,21187,39943,25932,64.92,82.72,84.0,108,275,161,12,0.87,12.48,17.36,3.91,349,4983,6935,1560
2,control,None,None,morning,14611,27422,18702,68.20,82.44,81.0,113,242,149,11,0.49,13.02,13.91,4.06,134,3569,3815,1113
3,control,None,None,rest_evening,9964,16277,10394,63.86,83.34,90.0,108,261,163,12,0.56,12.79,18.88,3.27,91,2082,3073,532
4,control,None,None,rest_morning,8736,13907,9441,67.89,86.37,94.0,111,219,149,8,0.77,9.94,17.98,2.91,107,1383,2501,405
5,test,viewed,close_click,afternoon,18578,34991,25935,74.12,90.11,60.0,108,249,170,8,0.46,7.21,15.61,2.23,160,2522,5463,780
6,test,viewed,close_click,evening,15335,28553,18703,65.50,83.02,57.0,111,259,159,12,0.91,12.01,17.06,4.03,261,3429,4870,1152
7,test,viewed,close_click,morning,10732,20439,13794,67.49,81.56,78.0,115,233,147,10,0.65,13.65,13.80,4.14,132,2789,2820,847
8,test,viewed,close_click,rest_evening,7120,11573,7374,63.72,83.64,55.0,98,263,162,12,0.49,12.82,19.26,3.00,57,1484,2229,347
9,test,viewed,close_click,rest_morning,5959,9363,6399,68.34,85.66,111.0,105,201,141,7,0.94,10.32,16.79,3.09,88,966,1572,289


In [89]:
session_sports.to_clipboard(index=False)

In [70]:
session_sports.to_csv('session_sports.csv',index=False)

In [71]:
ltr_segment_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-11'
    
),


merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type,
            segment.ltr_segment
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd
    LEFT JOIN 
            segment
        ON rr_net.customer_id=segment.customer_id

)

SELECT
        test_group,
        event_type,
        click_type,
        ltr_segment,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            ltr_segment,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

ltr_segment_sports=pd.read_sql(ltr_segment_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/657598224.py:203: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ltr_segment_sports=pd.read_sql(ltr_segment_query,conn)


In [72]:
ltr_segment_sports

,test_group,event_type,click_type,ltr_segment,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,HH,2092,3980,2457,61.73,86.71,90.0,112.0,193.0,143.0,8.0,1.08,9.20,24.35,3.02,43,366,969,120
1,control,None,None,PHH,41446,114664,80138,69.89,86.43,84.0,107.0,255.0,161.0,10.0,0.55,10.04,16.12,2.97,635,11511,18488,3407
2,control,None,None,UNKNOWN,125,357,93,26.05,83.19,108.0,111.0,147.0,115.0,8.0,6.44,7.84,56.30,2.52,23,28,201,9
3,control,None,None,None,6,6,6,100.00,100.00,NaN,NaN,NaN,NaN,8.0,0.00,0.00,0.00,0.00,0,0,0,0
4,test,viewed,close_click,HH,1634,3201,1924,60.11,84.82,49.0,99.0,227.0,157.0,9.0,2.37,8.59,23.96,4.22,76,275,767,135
5,test,viewed,close_click,PHH,29901,81419,56935,69.93,86.29,65.0,107.0,247.0,157.0,10.0,0.61,10.09,15.94,2.99,500,8218,12982,2434
6,test,viewed,close_click,UNKNOWN,83,227,58,25.55,80.18,144.0,113.0,107.0,110.0,8.0,5.29,9.69,54.19,4.85,12,22,123,11
7,test,viewed,play_click,HH,120,225,113,50.22,81.33,114.0,89.0,165.0,128.0,10.0,2.67,8.89,30.67,7.11,6,20,69,16
8,test,viewed,play_click,PHH,1687,3121,1967,63.02,83.08,36.0,96.0,189.0,124.0,10.0,0.51,13.20,19.61,3.20,16,412,612,100
9,test,viewed,play_click,UNKNOWN,10,32,1,3.13,34.38,70.0,87.0,50.0,70.0,5.0,18.75,18.75,28.13,28.13,6,6,9,9


In [73]:
retention_segment_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-11'
    
),


merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type,
            segment.retention_segment
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd
    LEFT JOIN 
            segment
        ON rr_net.customer_id=segment.customer_id

)

SELECT
        test_group,
        event_type,
        click_type,
        retention_segment,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            retention_segment,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

retention_segment_sports=pd.read_sql(retention_segment_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/1887586811.py:203: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  retention_segment_sports=pd.read_sql(retention_segment_query,conn)


In [74]:
retention_segment_sports

,test_group,event_type,click_type,retention_segment,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,DORMANT,5676,9929,6415,64.61,86.48,113.0,119.0,240.0,169.0,9.0,0.65,10.02,21.38,2.84,65,995,2123,282
1,control,None,None,ELITE,8758,41460,29426,70.97,86.23,62.0,98.0,242.0,149.0,10.0,0.44,10.48,14.90,2.85,181,4345,6177,1181
2,control,None,None,GOLD,10313,21589,15025,69.60,86.67,86.0,113.0,272.0,170.0,10.0,0.67,9.61,16.60,3.03,144,2075,3583,654
3,control,None,None,HH,1431,2810,1808,64.34,87.15,107.0,117.0,233.0,163.0,8.0,0.78,8.97,22.06,3.10,22,252,620,87
4,control,None,None,INACTIVE,384,681,311,45.67,83.26,71.0,90.0,42.0,55.0,7.0,1.76,10.87,37.00,4.11,12,74,252,28
5,control,None,None,PLATINUM,12023,31798,22528,70.85,86.88,83.0,108.0,262.0,163.0,9.0,0.50,9.73,15.63,2.88,160,3093,4970,916
6,control,None,None,PRIME,445,2373,1654,69.70,82.22,92.0,100.0,221.0,128.0,11.0,2.40,11.34,12.26,4.05,57,269,291,96
7,control,None,None,SILVER,4508,8004,5428,67.82,86.33,68.0,124.0,287.0,183.0,10.0,0.46,9.67,18.00,3.54,37,774,1441,283
8,control,None,None,UNKNOWN,125,357,93,26.05,83.19,108.0,111.0,147.0,115.0,8.0,6.44,7.84,56.30,2.52,23,28,201,9
9,control,None,None,None,6,6,6,100.00,100.00,NaN,NaN,NaN,NaN,8.0,0.00,0.00,0.00,0.00,0,0,0,0


In [75]:
service_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-11'
    
),


merge as(
    select exp_category.*,
            rr_net.service_category,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type,
            segment.retention_segment
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd
    LEFT JOIN 
            segment
        ON rr_net.customer_id=segment.customer_id

)

SELECT
        test_group,
        event_type,
        click_type,
        service_category,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            service_category,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

service_sports=pd.read_sql(service_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/728547421.py:210: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  service_sports=pd.read_sql(service_query,conn)


In [76]:
service_sports

,test_group,event_type,click_type,service_category,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,auto,21232,46041,31648,68.74,87.59,75.0,115,241,173,7,1.00,7.88,18.78,3.53,460,3627,8647,1623
1,control,None,None,bike,27130,63105,45764,72.52,87.73,97.0,98,251,150,11,0.31,9.50,14.47,2.46,194,5996,9131,1552
2,control,None,None,cab,5089,9861,5282,53.56,72.69,102.0,117,301,156,19,0.48,23.14,19.07,3.66,47,2282,1880,361
3,test,viewed,close_click,auto,15504,33595,23026,68.54,87.30,51.0,108,242,169,7,1.16,7.89,18.69,3.64,389,2649,6279,1224
4,test,viewed,close_click,bike,19189,44050,32055,72.77,87.70,85.0,101,238,147,11,0.34,9.50,14.18,2.45,150,4183,6246,1078
5,test,viewed,close_click,cab,3712,7202,3836,53.26,72.06,96.0,119,286,159,19,0.68,23.37,18.70,3.86,49,1683,1347,278
6,test,viewed,play_click,auto,715,1218,730,59.93,83.83,22.0,105,180,133,8,1.81,10.10,23.81,4.27,22,123,290,52
7,test,viewed,play_click,bike,1160,1880,1241,66.01,85.21,172.0,82,170,111,11,0.32,11.22,18.40,3.24,6,211,346,61
8,test,viewed,play_click,cab,151,281,110,39.15,58.36,NaN,105,308,133,22,0.00,37.01,19.22,4.63,0,104,54,13
9,test,viewed,None,auto,494,847,561,66.23,90.20,65.0,134,108,111,7,1.77,5.55,23.97,2.48,15,47,203,21


In [93]:
cluster_level=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),



merge as(
    select exp_category.*,
            rr_net.service_category,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.cluster_name,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
            
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd
    
)

SELECT
        test_group,
        event_type,
        click_type,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            cluster_name,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

cluster_sports=pd.read_sql(cluster_level,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/1617297090.py:209: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cluster_sports=pd.read_sql(cluster_level,conn)


In [95]:
cluster_sports=cluster_sports[cluster_sports['click_type']=='play_click']

In [96]:
cluster_sports

,test_group,event_type,click_type,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
24,test,viewed,play_click,Banjara Hills,45,68,35,51.47,77.94,119.0,148.0,227,148,13,1.47,14.71,26.47,5.88,1,10,18,4
25,test,viewed,play_click,Bodduppal,48,72,48,66.67,87.50,NaN,7.0,71,57,7,0.00,8.33,20.83,4.17,0,6,15,3
26,test,viewed,play_click,Gandipet,32,46,21,45.65,71.74,NaN,187.0,478,311,11,0.00,15.22,23.91,13.04,0,7,11,6
27,test,viewed,play_click,Hitech City,97,135,85,62.96,79.26,NaN,118.0,197,140,10,0.00,18.52,15.56,2.22,0,25,21,3
28,test,viewed,play_click,Kothaguda 2,40,59,35,59.32,72.88,NaN,116.0,92,98,10,0.00,16.95,13.56,10.17,0,10,8,6
29,test,viewed,play_click,Kukatpally 3,54,102,48,47.06,79.41,114.0,22.0,258,146,10,4.90,5.88,31.37,9.80,5,6,32,10
30,test,viewed,play_click,Manikonda,36,54,32,59.26,88.89,NaN,153.0,48,88,13,0.00,11.11,27.78,0.00,0,6,15,0
31,test,viewed,play_click,Marredpally,30,37,28,75.68,91.89,NaN,78.0,229,175,15,0.00,5.41,16.22,2.70,0,2,6,1
32,test,viewed,play_click,Mehedipatnam,43,65,38,58.46,81.54,NaN,96.0,160,124,11,0.00,15.38,23.08,3.08,0,10,15,2
33,test,viewed,play_click,Mothinagar,31,45,27,60.00,82.22,NaN,91.0,159,95,9,0.00,17.78,22.22,0.00,0,8,10,0


In [97]:
cluster_sports.to_clipboard(index=False)

In [102]:
city_cluster_level=f"""


with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
)



SELECT
        
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            rr_net
        where 
            yyyymmdd is not null
           
        GROUP BY 1
        )
    ORDER BY 1

"""

city_cluster_level_sports=pd.read_sql(city_cluster_level,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/1215307344.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  city_cluster_level_sports=pd.read_sql(city_cluster_level,conn)


In [103]:
city_cluster_level_sports

,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,Banjara Hills,26839,48199,30155,62.56,81.51,58,112,292,174,15,1.10,12.42,18.59,4.95,531,5987,8962,2385
1,Bodduppal,17235,29578,23840,80.60,94.70,46,104,242,182,7,0.26,3.54,13.58,1.50,78,1046,4017,444
2,Gandipet,5888,11493,6298,54.80,74.66,97,126,270,167,10,1.70,16.91,19.46,6.72,195,1943,2236,772
3,Hitech City,48650,93937,58193,61.95,79.71,61,111,236,152,11,0.98,13.90,17.46,5.40,919,13058,16399,5070
4,Kothaguda 2,15429,28542,18166,63.65,81.43,88,117,265,164,11,0.77,12.87,17.41,4.92,220,3673,4969,1404
5,Kukatpally 3,23890,39811,29656,74.49,91.20,29,101,248,174,9,0.46,5.73,16.24,2.60,182,2282,6466,1037
6,Manikonda,15328,28261,19683,69.65,87.18,31,113,254,169,10,0.27,9.29,17.20,3.25,76,2625,4862,919
7,Marredpally,14962,24039,17707,73.66,90.17,37,105,272,180,10,0.21,7.54,16.12,2.06,51,1813,3874,495
8,Mehedipatnam,26973,48199,33653,69.82,87.44,59,105,270,176,11,0.89,7.80,17.25,3.86,430,3760,8315,1861
9,Mothinagar,14314,26627,17350,65.16,83.74,98,115,279,178,11,0.99,10.76,18.26,4.50,264,2864,4863,1199


In [104]:
city_cluster_level_sports.to_clipboard(index=False)

In [105]:
cluster_session_level=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchSportsNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9e70d8e21590001b635ac','66d9e97fdac1090001c05d83')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240915'
    and event_props_campaign_id = '1725971334'

),



merge as(
    select exp_category.*,
            rr_net.service_category,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.cluster_name,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
            
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on view_tbl.user_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd
    
)

SELECT
        test_group,
        event_type,
        click_type,
        time_period,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4,5
        )
    ORDER BY 1,2,3,4,5

"""

cluster_session_level_sports=pd.read_sql(cluster_session_level,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/290345653.py:217: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cluster_session_level_sports=pd.read_sql(cluster_session_level,conn)


In [110]:
cluster_session_level_sports

,test_group,event_type,click_type,time_period,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
120,test,viewed,play_click,afternoon,Banjara Hills,17,24,17,70.83,83.33,NaN,148.0,101.0,137.0,9.0,0.00,8.33,12.50,8.33,0,2,3,2
121,test,viewed,play_click,afternoon,Bodduppal,18,20,16,80.00,90.00,NaN,6.0,984.0,57.0,7.0,0.00,10.00,10.00,0.00,0,2,2,0
122,test,viewed,play_click,afternoon,Gandipet,13,17,8,47.06,76.47,NaN,387.0,525.0,478.0,7.0,0.00,11.76,23.53,11.76,0,2,4,2
123,test,viewed,play_click,afternoon,Hitech City,38,49,35,71.43,91.84,NaN,104.0,196.0,135.0,8.0,0.00,8.16,18.37,0.00,0,4,9,0
124,test,viewed,play_click,afternoon,Kothaguda 2,10,11,8,72.73,90.91,NaN,136.0,344.0,136.0,6.0,0.00,9.09,18.18,0.00,0,1,2,0
125,test,viewed,play_click,afternoon,Kukatpally 3,23,41,18,43.90,85.37,114.0,22.0,290.0,149.0,10.0,4.88,2.44,39.02,7.32,2,1,16,3
126,test,viewed,play_click,afternoon,Manikonda,17,22,14,63.64,86.36,NaN,13.0,169.0,144.0,13.0,0.00,13.64,22.73,0.00,0,3,5,0
127,test,viewed,play_click,afternoon,Marredpally,13,14,12,85.71,92.86,NaN,NaN,4240.0,4240.0,17.0,0.00,0.00,7.14,7.14,0,0,1,1
128,test,viewed,play_click,afternoon,Mehedipatnam,14,20,11,55.00,70.00,NaN,96.0,133.0,96.0,15.0,0.00,30.00,15.00,0.00,0,6,3,0
129,test,viewed,play_click,afternoon,Mothinagar,13,19,13,68.42,94.74,NaN,20.0,46.0,46.0,8.0,0.00,5.26,26.32,0.00,0,1,5,0


In [109]:
cluster_session_level_sports=cluster_session_level_sports[cluster_session_level_sports['click_type']=='play_click']

In [111]:
cluster_session_level_sports.to_clipboard(index=False)

In [107]:
city_cluster_level_benchmark=f"""


with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
)



SELECT
        time_period,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            rr_net
        where 
            yyyymmdd is not null
           
        GROUP BY 1,2
        )
    ORDER BY 1,2

"""

city_cluster_level_sports_benchmark=pd.read_sql(city_cluster_level_benchmark,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_49271/2430106976.py:139: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  city_cluster_level_sports_benchmark=pd.read_sql(city_cluster_level_benchmark,conn)


In [108]:
city_cluster_level_sports_benchmark

,time_period,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,afternoon,Banjara Hills,12097,17456,12200,69.89,87.89,51,113,285,184,12,0.52,8.81,17.71,2.78,90,1538,3091,486
1,afternoon,Bodduppal,7836,11007,9144,83.07,96.24,56,108,253,205,6,0.09,2.54,12.69,1.13,10,280,1397,124
2,afternoon,Gandipet,2627,4144,2443,58.95,78.21,82,120,257,160,9,1.35,15.18,18.99,5.26,56,629,787,218
3,afternoon,Hitech City,19223,27480,20754,75.52,92.77,53,90,225,165,8,0.52,4.97,16.90,1.75,142,1366,4644,480
4,afternoon,Kothaguda 2,6412,9064,6757,74.55,91.38,100,102,257,183,9,0.50,5.78,16.49,2.34,45,524,1495,212
5,afternoon,Kukatpally 3,10241,14279,11130,77.95,93.99,69,96,264,191,8,0.29,4.15,15.64,1.57,42,592,2233,224
6,afternoon,Manikonda,6549,9624,7187,74.68,91.41,26,101,246,173,9,0.25,6.11,16.51,2.23,24,588,1589,215
7,afternoon,Marredpally,6938,9748,7139,73.24,89.55,60,106,299,190,11,0.19,7.77,15.99,2.48,19,757,1559,242
8,afternoon,Mehedipatnam,12660,19227,13793,71.74,89.17,34,102,274,182,11,0.68,6.90,17.09,3.26,130,1326,3286,627
9,afternoon,Mothinagar,5884,8567,6220,72.60,90.76,74,106,277,187,10,0.93,6.16,17.87,2.15,80,528,1531,184


In [112]:
city_cluster_level_sports_benchmark.to_clipboard(index=False)

In [19]:
click_bucket=f"""
with olf as (

    SELECT
        yyyymmdd,
        customer_id,
        order_id,
        order_requested_epoch,
        customer_cancelled_epoch,
        dropped_epoch,
        expired_epoch,
        rider_accepted_epoch
        
    FROM 
        orders.order_logs_fact
    WHERE
        yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        
    ),




click as(
    select
        yyyymmdd,
        profile_identity as user_id,
        epoch as click_epoch
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_wzrk_c2a='Button 2'
    and event_props_campaign_id = '1725971334'

),


merge AS (
    
    SELECT 
        olf.*,
        click_epoch,
        (click_epoch / 1000 - order_requested_epoch / 1000) time_to_click
        --row_number() over(partition by olf.yyyymmdd,olf.customer_id,olf.order_id order by click_epoch) seq_no
    FROM 
        olf
    JOIN 
        click
        on olf.yyyymmdd = click.yyyymmdd
        and olf.customer_id = click.user_id
        and olf.order_requested_epoch < click.click_epoch
        and (olf.customer_cancelled_epoch > click.click_epoch or dropped_epoch > click.click_epoch or 
        expired_epoch > click.click_epoch or rider_accepted_epoch> click.click_epoch)
    )
    
    
   
    
    
    SELECT
        
        case 
        when time_to_click <= 15 then 'a. 15 Sec'
        when time_to_click between 16 and 30 then 'b. 16-30 Sec'
        when time_to_click between 31 and 60 then 'c. 30-60 Sec'
        when time_to_click between 61 and 120 then 'd. 1-2 min'
        when time_to_click between 121 and 240 then 'e. 3-4 min'
        when time_to_click > 240 then 'f. 4+ min' 
        else 'check' end as click_bucket,
        count(distinct customer_id) customers
    FROM 
        merge
    -- WHERE 
    --     --seq_no = 1
    GROUP BY 1
    ORDER BY 1

"""
click_bucket_df = pd.read_sql(click_bucket,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62376/650900096.py:82: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_bucket_df = pd.read_sql(click_bucket,conn)


In [20]:
click_bucket_df

,click_bucket,customers
0,a. 15 Sec,1737
1,b. 16-30 Sec,274
2,c. 30-60 Sec,123
3,d. 1-2 min,81
4,e. 3-4 min,35
5,f. 4+ min,33


In [21]:
click_bucket_df.customers.sum()

2283

In [22]:
2178-2283

-105

In [24]:
1737-105


1632

In [23]:
click_bucket_df.to_clipboard(index=False)

In [117]:
total_customers = click_bucket_df['customers'].sum()

# Calculate the percentage of customers for each click_bucket
click_bucket_df['customer_percentage'] = (click_bucket_df['customers'] / total_customers) * 100

# Round the percentage to 2 decimal places
click_bucket_df['customer_percentage'] = click_bucket_df['customer_percentage'].round(2)


In [119]:
click_bucket_df.to_clipboard(index=False)